### Vector Search 

*Note*: vector search adds complexity to the retrieval steps, hence it is preferable to start with a normal text search (as in lessons 1 and 2), and only then consider using vector search. 

`query = "I just discovered the course, can I still join?"`

**Text Search**: decompose query into meaningful components (discover, course, join) and look for documents containing at least 1, 2 , 3, or more of the meaningful tems (TF-IDF, BM25, ...)

But imagine the query: 

`query = "I just found out about the program, can I still enroll?"`

Semantically, the query is the same, but meaningful terms in it are different (find, program, enroll) and might lead to different retrieved documents (or no documents at all).  **Vector Search** helps overcoming this issue (semantic search).  

In [6]:
q1 = "Can I still join the course after the start date?"
q2 = "I just found out about the program, can I still enroll?"

In [2]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
v1 = model.encode(q1)
v1.shape

In [10]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [ ]:
v1.dot(dv)    # vector multiplication to measure similarity --> COSINE SIMILARITY as the two vectors are normalized 

np.float32(0.32332397)

In [15]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
v2.dot(dv)

np.float32(0.019730574)

In [16]:
from ingest import load_faq_data 
documents = load_faq_data() 

In [17]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [18]:
texts = [] 
for doc in documents: 
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [19]:
texts[10]

'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [20]:
len(texts)

1346

In [21]:
from tqdm.auto import tqdm
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1346

In [ ]:
v1.dot(vectors[10])

np.float32(0.33153272)

In [23]:
# Compute all documents' similarities using matrix multiplication 
import numpy as np 
X = np.array(vectors)
X.shape

(1346, 384)

In [24]:
X

array([[-0.02670618, -0.12245757,  0.01594413, ..., -0.00230654,
        -0.11218394, -0.02365559],
       [-0.04383259, -0.09589145, -0.01538173, ...,  0.11842171,
        -0.00699413,  0.01887378],
       [-0.08896548, -0.06128178,  0.00775603, ...,  0.0405971 ,
         0.00479277, -0.02745943],
       ...,
       [-0.03652925,  0.01415426, -0.06838644, ...,  0.04316786,
         0.08105537, -0.02148626],
       [-0.13091585, -0.06990602, -0.0093188 , ..., -0.00044347,
        -0.01285731,  0.01426919],
       [-0.07984784,  0.01926981,  0.02544978, ..., -0.03368027,
        -0.01884026,  0.05837054]], shape=(1346, 384), dtype=float32)

In [25]:
scores = X.dot(v1)

In [26]:
scores 

array([ 0.48740587,  0.26981184,  0.762941  , ..., -0.08637965,
        0.03759791, -0.03037035], shape=(1346,), dtype=float32)

In [28]:
# Find the most similar document 
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [29]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [30]:
# Top-5 scores 
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
scores[top5]

array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

In [31]:
for idx in top5: 
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

In [32]:
# Alternative to [::-1]: 
# np.argsort(-scores) # -> first indices are those with highest positive scores

With minsearch: 

In [34]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [36]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the for

**Rag**:  we will replace the text-based RAGBase class with a Vector-Based search. 

In [38]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [39]:
from rag_helper import RAGBase

class RAGVector(RAGBase):
    
    def __init__(self, embedder, **kwargs): 
        super().__init__(**kwargs)
        self.embedder = embedder 

    def search(self, query, num_results=5): 
        query_vector = self.embedder.encode(query) 
        filter_dict = {'course': self.course} 

        return self.index.search(
            query_vector, 
            num_results=num_results, 
            filter_dict=filter_dict, 
        ) 


In [40]:
vector_assistant = RAGVector(
    embedder=model, 
    index=vindex, 
    llm_client=openai_client, 
)

In [43]:
query = "the program has already begun, can I still sign up?"
vector_assistant.rag(query)

'Yes — you can still join even if the program has already begun. If you want a certificate, make sure to submit your project while submissions are still open.'

So far `minsearch` has implemented NN (nearest neighbors) algorithm to find closest documents. This calculates similarities across all possible documents: it is exact but time-consuming. 
Conversely, there exists an algorithm, Approximate NN (ANN) that only calculates similarities in an area closer to the Query vector. Approximate results, but faster. 

`sqlitesearch` allows persisting documents to db, in a sqlite db. 

In [44]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

sqlitesearch supports three ANN modes:

* lsh (default): up to 100K vectors, random hyperplane projections
* ivf: 10K-500K vectors, K-means clustering
* hnsw: 10K-1M+ vectors, proximity graph (highest recall)

In [45]:
vs_index.fit(vectors, documents)

In [46]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [48]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [49]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [50]:
vs_index.close() # close connection 

**PGVector**

See PGVector notebook.

**ONNX**. Offers a lighter runtime environment for pytorch models. 

```{bash}
mkdir llm-zoomcamp-onnx && cd llm-zoomcamp-onnx
uv init --no-workspace
uv add onnxruntime tokenizers numpy tqdm minsearch
uv add --dev huggingface-hub jupyter
```

`huggingface-hub` is only needed to download the model. At runtime we'll need onnxruntime, tokenizers, and numpy.

Register a kernel for the project:

```{bash}
uv run python -m ipykernel install --user --name llm-zoomcamp-onnx --display-name "llm-zoomcamp-onnx"
```

Download model (embed/download.py): 

```uv run python download.py```

Which will create: 

```
models/
  Xenova/
    all-MiniLM-L6-v2/
      tokenizer.json
      model.onnx
```

Add to .gitignore: 

`models/`

To generate embeddings, use `embed/embedder.py`  --> it does the following: 
1. tokenize
2. run ONNX model (execute model graph on cpu)
3. mean pooling (average token embeddings weighted by attention mask)
4. normalize (divide by L2 norm so vectors are comparable)

Compare following results with those obtained above (pytorch): 

In [ ]:
from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

In [ ]:
# Compute similarities 
v1.dot(dv)
v2.dot(dv)

In [ ]:
from ingest import load_faq_data
documents = load_faq_data()

In [ ]:
texts = [doc["question"] + " " + doc["answer"] for doc in documents]
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

In [ ]:
query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

Result is the same, pipeline ~33x lighter. 

All of these work with the same code - just change the model name in download.py and the path in Embedder():

* Xenova/all-MiniLM-L6-v2 (384d) - best small general-purpose
* Xenova/all-MiniLM-L12-v2 (384d) - better quality, slower
* Xenova/paraphrase-MiniLM-L6-v2 (384d) - paraphrase detection
* Xenova/paraphrase-multilingual-MiniLM-L12-v2 (384d) - multilingual
* Xenova/multilingual-e5-small (384d) - multilingual retrieval
* Xenova/multilingual-e5-base (768d) - stronger multilingual
* Xenova/bge-small-en-v1.5 (384d) - strong retrieval
* Xenova/bge-base-en-v1.5 (768d) - stronger retrieval
* Xenova/gte-small (384d) - lightweight modern model
* Xenova/gte-base (768d) - stronger GTE

To use a different model, add it to `download.py` (*), run script, and update path: 

In [ ]:
embed = Embedder("models/Xenova/bge-base-en-v1.5")
vectors = embed.encode("your text here")
print(vectors.shape)

(*) = actually just specify it as argument of the `download` function, `download("Xenova/all-MiniLM-L6-v2")` [just make sure model file name/extension matches those in download.py]

Since the runtime only depends on onnxruntime, tokenizers, and numpy, you can deploy this in minimal environments:

* small Docker images
* serverless functions
* edge devices